# Cell Embedding Model (4/2, Adding Grid Search Cross Validation)
This notebook:
- Builds graphs for each of the reigons within dataset using nearest neigbors, with each region constitution a connected component
- Trains a single GCN for the cell-type classification task on these graph
- Uses the soft logits from the final hidden layer (d=25 for the 25 cell types) as embeddings of every cell in the dataset
- Saves a file with all of the cell embeddings
- Trains a NN (Autoencoder) to reduce these embeddings to d=3 and then recover th
- Normalizes these outputs to RGB values and paints a PNG image for each region

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import os

# PyTorch Geometric
!pip install torch_geometric
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv

# For kNN
from sklearn.neighbors import NearestNeighbors

torch.manual_seed(42)
np.random.seed(42)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.4 MB/s eta 0:00:00


## Data Preparation
Below, we construct a class which represents each individual region and builds a graph using the nearest neighbors of each cell.
Afterwards, we use this class to create objects for each of the unique regions in our dataset.

In [ ]:
"""
RegionDataset Class
This class provides a useful abstraction for storing individual regions, enabling us to easily go between the graph represenation and the typical
"""
class RegionDataset(Dataset):
    def __init__(self, features_df, unique_region, labels_encoded, x_positions, y_positions, k_neighbors=5):
        """
        features_df: (DataFrame) feature columns for all cells
        unique_region: (Series or Array) region ID per row
        labels_encoded: (Array) numeric label (cell type) per row
        x_positions: (Array) x-coordinate per row (for later saving)
        y_positions: (Array) y-coordinate per row (for later saving)
        k_neighbors: (int) k for nearest neighbors
        """
        super().__init__()
        self.region_list = np.unique(unique_region)
        self.features_df = features_df.values  # Convert to NumPy up front
        self.unique_region = unique_region
        self.labels_encoded = labels_encoded
        self.x_positions = x_positions
        self.y_positions = y_positions
        self.k = k_neighbors

        # Pre-compute k-NN graphs for all regions
        self.edge_indices_dict = {}
        for region_id in self.region_list:
            region_mask = (self.unique_region == region_id)
            region_features = self.features_df[region_mask]

            knn = NearestNeighbors(n_neighbors=self.k)
            knn.fit(region_features)
            distances, indices = knn.kneighbors(region_features)

            edge_indices = []
            for i, neighbors in enumerate(indices):
                for nbr in neighbors:
                    if i != nbr:
                        edge_indices.append([i, nbr])

            self.edge_indices_dict[region_id] = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()

    def len(self):
        return len(self.region_list)

    def get(self, idx):
      region_id = self.region_list[idx]
      region_mask = (self.unique_region == region_id)

      region_features = self.features_df[region_mask]           # shape [num_nodes, in_channels]
      region_labels = self.labels_encoded[region_mask]          # shape [num_nodes]
      region_xpos = self.x_positions[region_mask]               # shape [num_nodes]
      region_ypos = self.y_positions[region_mask]               # shape [num_nodes]

      # Retrieve pre-computed edge indices
      edge_index = self.edge_indices_dict[region_id]

      # Convert to tensors
      x = torch.tensor(region_features, dtype=torch.float32)
      y = torch.tensor(region_labels, dtype=torch.long)

      # We'll store x_pos, y_pos for later retrieval
      data = Data(x=x, edge_index=edge_index, y=y)
      data.x_pos = torch.tensor(region_xpos, dtype=torch.float32)
      data.y_pos = torch.tensor(region_ypos, dtype=torch.float32)
      data.region_id = region_id  # just keep track of the region name/ID

      return data.to(device)



In [ ]:
# Set pandas to show all columns for debugging
pd.set_option('display.max_columns', None)

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/1024x1024_dataset.csv', index_col=0)

# Parse columns
unique_region = df["unique_region"].values
labels = df["Cell Type"].values

# Cell positions
x_positions = df['x'].to_numpy()
y_positions = df['y'].to_numpy()
print(x_positions)

# Drop everything not used in GNN features
drop_cols = [
    "Cell Type", "unique_region", "donor", "array", "Xcorr", "Ycorr",
    "Tissue_location", "tissue", "region", "OLFM4", "FAP", "CD25",
    "CK7", "MUC6", "Cell Type em", "Cell subtype", "Neighborhood",
    "Neigh_sub", "Neighborhood_Ind", "NeighInd_sub", "Community",
    "Major Community", "Tissue Segment", "Tissue Unit", "CollIV"
]
features_df = df.drop(columns=drop_cols)

features_df.head()

<ipython-input-7-4ed223b2dbee>:5: DtypeWarning: Columns (73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/1024x1024_dataset.csv', index_col=0)


[434 565 661 ... 723 227 232]


,SOX9,MUC1,CD31,Synapto,CD49f,CD15,CHGA,CDX2,ITLN1,CD4,CD127,Vimentin,HLADR,CD8,CD11c,CD44,CD16,BCL2,CD3,CD123,CD38,CD90,aSMA,CD21,NKG2D,CD66,CD57,CD206,CD68,CD34,aDef5,CD7,CD36,CD138,CD45RO,Cytokeratin,CD117,CD19,Podoplanin,CD45,CD56,CD69,Ki67,CD49a,CD163,CD161,x,y
MUC2,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
-0.303994,-0.163727,-0.587608,-0.212903,0.164173,-0.664863,0.049305,0.003616,-0.377532,-0.450794,-0.412798,-0.242456,-0.443621,-0.305035,-0.215787,-0.090214,-0.790193,-0.215371,-0.512302,-0.304414,-0.237758,-0.085275,-0.444684,-0.294004,-0.238059,-0.174056,-0.411411,12.815198,-0.310213,-0.490891,-0.331497,-0.128173,-0.306406,-0.228822,-0.247633,-0.334483,-0.281628,-0.229013,-0.136007,-0.126372,-0.226603,-0.282104,-0.337309,-0.328652,-0.736535,-0.304606,-0.188373,434,382
-0.301927,-0.491706,-0.500804,-0.243205,-0.142568,-0.664861,-0.182627,-0.117573,-0.182754,-0.236199,-0.414171,-0.203436,-0.465625,-0.472001,-0.215740,-0.242785,-0.790192,-0.282328,-0.325940,-0.415133,-0.373112,-0.296364,-0.461532,-0.272272,-0.217521,-0.160158,-0.334227,-0.392993,-0.309681,-0.362089,-0.302879,-0.115029,-0.085223,-0.231418,-0.249979,-0.098562,-0.403709,-0.229013,-0.160003,-0.272051,-0.551045,0.063713,-0.381561,-0.311211,-0.736535,-0.324616,-0.181808,565,465
-0.302206,-0.547234,-0.510705,-0.235309,-0.217185,-0.622758,-0.296486,-0.091504,-0.268055,-0.355383,-0.078504,-0.197512,-0.484929,-0.404415,-0.200173,-0.262091,-0.106529,-0.240808,-0.501271,-0.294345,-0.364049,-0.322098,-0.471427,-0.293557,-0.227081,-0.134822,-0.384489,5.836004,-0.311383,0.377784,-0.234879,-0.116748,-0.254280,-0.060062,-0.246992,-0.123830,-0.520802,3.052579,-0.172104,-0.207255,-0.289779,-0.287262,-0.317390,-0.225915,-0.671212,-0.291291,-0.190268,661,355
-0.304219,-0.613068,-0.584499,-0.243757,-0.266696,-0.658449,-0.299027,-0.121460,-0.345381,-0.450792,-0.446967,-0.213501,-0.488082,-0.473116,-0.210042,-0.319812,-0.347987,-0.275794,-0.512305,-0.415086,-0.367479,-0.322098,-0.526044,-0.294004,-0.238060,-0.157920,-0.411412,2.774781,-0.311382,-0.207540,-0.311923,-0.123919,-0.306644,-0.087760,-0.261261,-0.351568,-0.522322,0.774255,-0.176094,-0.270622,-0.490348,-0.296769,-0.340400,-0.275875,-0.733125,-0.329447,-0.201727,826,267
-0.294644,-0.615593,-0.570580,-0.247548,-0.042246,-0.642230,-0.299031,-0.121458,-0.377533,-0.450797,-0.446969,-0.242456,-0.488082,-0.473116,-0.215786,-0.319814,-0.790191,-0.273713,-0.512299,-0.415133,-0.395929,-0.322099,-0.526044,-0.229718,-0.238059,-0.164432,-0.411412,-0.444966,-0.311382,-0.490890,-0.359675,-0.128175,-0.306646,-0.238008,-0.258301,-0.351567,-0.522323,-0.229013,-0.106623,-0.320673,-0.551044,0.062078,-0.384764,-0.323847,-0.700788,-0.329448,-0.201727,739,439


In [ ]:
# Encode labels
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

# Create dataset
dataset = RegionDataset(
    features_df=features_df,
    unique_region=unique_region,
    labels_encoded=labels_encoded,
    x_positions=x_positions,
    y_positions=y_positions,
    k_neighbors=20  # need to put at the highest setting that will appear in the grid search later
)

## GCN Model
Here, we have a straightforward 2-layer GCN. On each side of the 2 convolutional layers, there are fully connected layers for matching the dimension of the input and desired output. ReLU activation functions between each layer ensure the network is able to model nonlinear relationships and avoid the vanishing gradient problem.\\

For classification, set out_dim equal to the number of classes (25 for the intestine dataset).


In [ ]:
class GNNModel(nn.Module):
    def __init__(self, in_channels, hidden_dim, out_dim):
        super().__init__()
        self.pre_fc = nn.Linear(in_channels, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.post_fc = nn.Linear(hidden_dim, out_dim)
        self.to(device)

    def forward(self, x, edge_index):
        # 1) Pre-process
        x = self.pre_fc(x)
        x = torch.relu(x)
        # 2) GCN Layers
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.conv2(x, edge_index)
        x = torch.relu(x)
        # 3) Output
        x = self.post_fc(x)
        return x


def train_one_epoch(model, loader, criterion, optimizer):
    """
    Runs one epoch of training and returns total or average loss.
    """
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        loss = criterion(out, batch.y.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    """
    Evaluates the model on a validation/test loader and returns accuracy.
    """
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            out = model(batch.x, batch.edge_index)
            preds = out.argmax(dim=1)
            correct += (preds == batch.y).sum().item()
            total += batch.y.size(0)
    return correct / total if total > 0 else 0



## GCN with Grid Search CV
Using a dataloader enables us to train in batches of multiple region-graphs. Each batch merges the adjancency of its constitutive graphs.




In [ ]:
from torch_geometric.data import Dataset
from torch.utils.data import Subset
from torch.nn import DataParallel
from sklearn.model_selection import KFold
import itertools

# Extract the unique region IDs for splitting
all_region_ids = dataset.region_list  # region_dataset.region_list is from your RegionDataset
all_region_ids = np.array(all_region_ids)  # for indexing

# We'll do a 5-fold split on the unique region IDs
kf = KFold(n_splits=5, shuffle=True, random_state=42)


# Grid Search Setup
param_grid = {
    "hidden_dim": [32, 64, 128],
    "lr": [1e-3, 1e-2],
    "weight_decay": [0, 1e-4],
    "k_neighbors": [5, 10, 20],  # originally was 5 or 10, now we add 20
    "batch_size": [5, 10],
    "epochs": [20, 50]  # we try 20 and 50
}

in_channels = dataset.features_df.shape[1]
num_classes = len(label_encoder.classes_)


def build_subset_dataset(base_dataset, subset_region_ids, k_neighbors):
    """
    Creates a RegionDataset with the same features but only the subset of region IDs
    and the new k_neighbors setting. This means we have to rebuild it, or you can
    store everything and filter in 'get()' if you prefer.
    """
    # Filter the global DataFrame (or original arrays) to only the subset of region IDs
    mask = np.isin(base_dataset.unique_region, subset_region_ids)
    # Re-create a RegionDataset with the same underlying data, but a restricted set of rows
    # and updated k_neighbors
    sub_dataset = RegionDataset(
        features_df=pd.DataFrame(base_dataset.features_df[mask]),
        unique_region=base_dataset.unique_region[mask],
        labels_encoded=base_dataset.labels_encoded[mask],
        x_positions=base_dataset.x_positions[mask],
        y_positions=base_dataset.y_positions[mask],
        k_neighbors=k_neighbors
    )
    return sub_dataset


best_config = None
best_avg_acc = 0.0

# Iterate over all combinations of hyperparameters
for (hidden_dim, lr, weight_decay, k_neighbors, batch_size, n_epochs) in itertools.product(
    param_grid["hidden_dim"],
    param_grid["lr"],
    param_grid["weight_decay"],
    param_grid["k_neighbors"],
    param_grid["batch_size"],
    param_grid["epochs"]
):
    fold_accuracies = []

    for train_idx, val_idx in kf.split(all_region_ids):
        # region IDs for this fold
        train_region_ids = all_region_ids[train_idx]
        val_region_ids = all_region_ids[val_idx]

        # Build dataset for train/val
        train_dataset = build_subset_dataset(dataset, train_region_ids, k_neighbors)
        val_dataset = build_subset_dataset(dataset, val_region_ids, k_neighbors)

        # Create DataLoaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Initialize GCN model
        model = GNNModel(in_channels=in_channels, hidden_dim=hidden_dim, out_dim=num_classes)
        if torch.cuda.device_count() > 1: # check if we can run in parallel
            model = DataParallel(model)
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss()

        # Train for n_epochs
        for epoch in range(n_epochs):
            train_one_epoch(model, train_loader, criterion, optimizer)

        # Evaluate on val set
        val_acc = evaluate(model, val_loader)
        fold_accuracies.append(val_acc)

    # Mean accuracy for all folds
    mean_acc = np.mean(fold_accuracies)
    print(f"[GCN Grid] h={hidden_dim}, lr={lr}, wd={weight_decay}, k={k_neighbors}, "
          f"bs={batch_size}, epochs={n_epochs} => val_acc={mean_acc:.4f}")

    # Track best
    if mean_acc > best_avg_acc:
        best_avg_acc = mean_acc
        best_config = {
            "hidden_dim": hidden_dim,
            "lr": lr,
            "weight_decay": weight_decay,
            "k_neighbors": k_neighbors,
            "batch_size": batch_size,
            "epochs": n_epochs
        }

print("Best config:", best_config, "with mean val accuracy=", best_avg_acc)



ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-8-17140ba59a3f>", line 86, in <cell line: 0>
    train_one_epoch(model, train_loader, criterion, optimizer)
  File "<ipython-input-7-ffe91ae7e5b8>", line 30, in train_one_epoch
    for batch in loader:
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 708, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 764, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
           ^^^^^^^^^^^^^^^^^^

## Extract and Save the Embeddings
We can use the previously trained GCN to get embeddings from the final linear layer for every cell in the dataset. We then save these as a csv

In [ ]:
model.eval()

all_rows = [] # empty list to store each row including its embedding
for data in dataset:
    # data is one region
    with torch.no_grad():
        # get embeddings from final linear layer
        embeddings = model(data.x, data.edge_index)  # shape [num_nodes_in_this_region, hidden_dim]

    embeddings = embeddings.cpu().numpy()
    x_pos = data.x_pos.cpu().numpy()
    y_pos = data.y_pos.cpu().numpy()
    region_id = data.region_id  # might be a string or int
    labels_encoded_region = data.y.cpu().numpy()

    # Store each node in a DataFrame-like row
    for i in range(embeddings.shape[0]):
        row_dict = {
            "region": region_id,
            "x_pos": x_pos[i],
            "y_pos": y_pos[i]
        }

        # Decode the numeric label to the actual cell-type string
        cell_type_str = label_encoder.inverse_transform([labels_encoded_region[i]])[0]
        row_dict["Cell Type"] = cell_type_str

        # Add each dimension of embedding (useful if we switch to use embeddings from hidden layer)
        for dim_idx in range(embeddings.shape[1]):
            row_dict[f"emb_{dim_idx}"] = embeddings[i, dim_idx]
        all_rows.append(row_dict)

node_embeddings_df = pd.DataFrame(all_rows)
node_embeddings_df.to_csv("cell_embeddings.csv", index=False)


## Dimensional Recution Network with Grid Search CV
We want to map the GCN's embeddings down to 3 dimensions for RGB-like representation, so our bottleneck dimension is 3. We accomplish this via an Autoencoder trained on MSE reconstruction.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, in_dim=25, bottleneck_dim=3, hidden_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, in_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

    def encode(self, x):
        return self.encoder(x)


def train_autoencoder(model, x_train, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    x_recon = model(x_train)
    loss = criterion(x_recon, x_train)
    loss.backward()
    optimizer.step()
    return loss.item()


def eval_autoencoder(model, x_val, criterion):
    model.eval()
    with torch.no_grad():
        x_recon = model(x_val)
        val_loss = criterion(x_recon, x_val).item()
    return val_loss


In [ ]:
# Dataset of all emebddings
emb_matrix = node_embeddings_df[[col for col in node_embeddings_df.columns if col.startswith("emb_")]].values # shape = [N, hidden_dim], N = total nodes from all graphs
emb_matrix_tensor = torch.tensor(emb_matrix, dtype=torch.float32)

In [ ]:
# Let's do a single train/validation split
N = emb_matrix_tensor.shape[0]
indices = np.arange(N)
np.random.shuffle(indices)

train_frac = 0.8
split_idx = int(train_frac * N)
train_indices = indices[:split_idx]
val_indices = indices[split_idx:]

x_train = emb_matrix_tensor[train_indices]
x_val = emb_matrix_tensor[val_indices]

# Set up the grid
import itertools

ae_param_grid = {
    "hidden_dim": [16, 32, 64],
    "lr": [1e-3, 1e-2],
    "weight_decay": [0, 1e-4]
}

best_ae_config = None
best_ae_val_loss = float("inf")

for (hidden_dim, lr, wd) in itertools.product(
    ae_param_grid["hidden_dim"],
    ae_param_grid["lr"],
    ae_param_grid["weight_decay"]
):
    # Build a new AE
    autoenc = Autoencoder(in_dim=25, bottleneck_dim=3, hidden_dim=hidden_dim)
    optimizer = optim.Adam(autoenc.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.MSELoss()

    # We can fix the number of epochs for AE or do early stopping. Let’s do, e.g., 20 epochs for each setting:
    epochs_ae = 20
    for epoch in range(epochs_ae):
        loss_train = train_autoencoder(autoenc, x_train, optimizer, criterion)
        # optional: track val loss each epoch if you want
        # loss_val = eval_autoencoder(autoenc, x_val, criterion)

    # Evaluate final model
    final_val_loss = eval_autoencoder(autoenc, x_val, criterion)

    print(f"[AE Grid] hidden_dim={hidden_dim}, lr={lr}, wd={wd} => val_loss={final_val_loss:.4f}")
    if final_val_loss < best_ae_val_loss:
        best_ae_val_loss = final_val_loss
        best_ae_config = {
            "hidden_dim": hidden_dim,
            "lr": lr,
            "weight_decay": wd
        }

print("Best AE config:", best_ae_config, "with val_loss=", best_ae_val_loss)


[AE] Epoch 1/20, Loss: 12.9454
[AE] Epoch 2/20, Loss: 12.0138
[AE] Epoch 3/20, Loss: 10.7865
[AE] Epoch 4/20, Loss: 9.2021
[AE] Epoch 5/20, Loss: 7.3281
[AE] Epoch 6/20, Loss: 5.4653
[AE] Epoch 7/20, Loss: 4.2257
[AE] Epoch 8/20, Loss: 4.0124
[AE] Epoch 9/20, Loss: 3.8146
[AE] Epoch 10/20, Loss: 3.1815
[AE] Epoch 11/20, Loss: 2.5307
[AE] Epoch 12/20, Loss: 2.1662
[AE] Epoch 13/20, Loss: 2.1091
[AE] Epoch 14/20, Loss: 2.2149
[AE] Epoch 15/20, Loss: 2.3198
[AE] Epoch 16/20, Loss: 2.3289
[AE] Epoch 17/20, Loss: 2.2338
[AE] Epoch 18/20, Loss: 2.0831
[AE] Epoch 19/20, Loss: 1.9398
[AE] Epoch 20/20, Loss: 1.8420


## Normalize Outputs to RBG and Save as CSV

In [ ]:
autoenc.eval()
with torch.no_grad():
    x_in = torch.tensor(emb_matrix, dtype=torch.float32)
    z_3d = autoenc.encode(x_in).cpu().numpy()  # shape [N, 3]

# z_3d can have arbitrary range; let's min-max scale each dimension
min_vals = z_3d.min(axis=0)
max_vals = z_3d.max(axis=0)

# Avoid zero-division if max == min
range_vals = (max_vals - min_vals)
range_vals[range_vals == 0] = 1e-9

scaled_3d = (z_3d - min_vals) / range_vals  # [0, 1]
rgb_3d = (scaled_3d * 255).astype(np.uint8) # [0..255]

# Add columns back to df
node_embeddings_df["R"] = rgb_3d[:, 0]
node_embeddings_df["G"] = rgb_3d[:, 1]
node_embeddings_df["B"] = rgb_3d[:, 2]

# Save to csv
node_embeddings_df.to_csv("node_embeddings_3d.csv", index=False)


In [ ]:
# To get these constants for image_decoder.ipynb
print(min_vals)
print(max_vals)

NameError: name 'min_vals' is not defined

## Save individual images as PNG

In [ ]:
import cv2  # or PIL

save_dir = "region_images"
os.makedirs(save_dir, exist_ok=True)

all_regions = node_embeddings_df["region"].unique()

for reg in all_regions:
    subset = node_embeddings_df[node_embeddings_df["region"] == reg]
    xs = subset["x_pos"].values
    ys = subset["y_pos"].values
    Rs = subset["R"].values
    Gs = subset["G"].values
    Bs = subset["B"].values

    # Create a blank image (adjust size if needed)
    # Example: 1024 x 1024, 3 channels
    img = np.zeros((1024, 1024, 3), dtype=np.uint8)

    for i in range(len(subset)):
        x_int = int(round(xs[i]))
        y_int = int(round(ys[i]))
        # Safety check
        if 0 <= x_int < 1024 and 0 <= y_int < 1024:
            img[y_int, x_int, 0] = Bs[i]  # OpenCV uses BGR order by default
            img[y_int, x_int, 1] = Gs[i]
            img[y_int, x_int, 2] = Rs[i]

    # Save the image
    save_path = os.path.join(save_dir, f"region_{reg}.png")
    cv2.imwrite(save_path, img)

    print(f"Saved image for region {reg} -> {save_path}")


Saved image for region B004_Ascending -> region_images/region_B004_Ascending.png
Saved image for region B004_Descending -> region_images/region_B004_Descending.png
Saved image for region B004_Descending - Sigmoid -> region_images/region_B004_Descending - Sigmoid.png
Saved image for region B004_Duodenum -> region_images/region_B004_Duodenum.png
Saved image for region B004_Ileum -> region_images/region_B004_Ileum.png
Saved image for region B004_Mid-jejunum -> region_images/region_B004_Mid-jejunum.png
Saved image for region B004_Proximal Jejunum -> region_images/region_B004_Proximal Jejunum.png
Saved image for region B004_Transverse -> region_images/region_B004_Transverse.png
Saved image for region B005_Ascending -> region_images/region_B005_Ascending.png
Saved image for region B005_Descending -> region_images/region_B005_Descending.png
Saved image for region B005_Descending - Sigmoid -> region_images/region_B005_Descending - Sigmoid.png
Saved image for region B005_Duodenum -> region_imag

In [ ]:
# Save Models
import torch

# Save the classification model's parameters
torch.save(model.state_dict(), "gnn_model.pth")

# Save the autoencoder's parameters
torch.save(autoenc.state_dict(), "autoencoder.pth")